# jaxcapse vs. CAMB 2.0.4 + CosmoRec

Load one trained TT emulator, evaluate one in-domain cosmology, and compare it with the CAMB worker used for generation. Both predictions are lensed $D_\ell$ in $\mu K^2$.

Run with the jaxcapse Poetry environment. The notebook finds the neighboring `tools/CAMB-cosmorec` and `emulator-zoo/Capse.jl/camb_mnuw0wacdm` checkouts in this workspace. Set `CAMB_COSMOREC_ROOT` or `CAMB_WORKER_ROOT` if those sources are elsewhere.

In [1]:
from pathlib import Path
import os
import sys

repo_root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "jaxcapse" / "__init__.py").is_file()), None)
if repo_root is not None:
    sys.path.insert(0, str(repo_root))

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import numpy as np
import jaxcapse

jaxcapse_root = Path(jaxcapse.__file__).resolve().parents[1]
workspace = jaxcapse_root.parent
camb_root = Path(os.environ.get("CAMB_COSMOREC_ROOT", workspace / "tools" / "CAMB-cosmorec")).resolve()
worker_root = Path(os.environ.get("CAMB_WORKER_ROOT", workspace / "emulator-zoo" / "Capse.jl" / "camb_mnuw0wacdm")).resolve()
assert (camb_root / "camb" / "camblib.so").is_file(), camb_root
assert (worker_root / "camb_worker.py").is_file(), worker_root
sys.path.insert(0, str(camb_root))
sys.path.insert(0, str(worker_root))
import camb_worker

configuration = camb_worker.backend_configuration()
assert configuration["camb_version"] == "2.0.4"
assert configuration["recombination_model"] == "CosmoRec"
print(f"CAMB {configuration['camb_version']} + {configuration['recombination_model']}")

CAMB 2.0.4 + CosmoRec


In [2]:
# [ln10As, ns, tau, H0, omega_b, omega_c, Mnu, w0, wa]; w0 + wa < -0.5.
params = np.array([3.044, 0.965, 0.054, 67.4, 0.02237, 0.12, 0.06, -1.0, 0.0], dtype=np.float64)
tt = jaxcapse.trained_emulators["camb_mnuw0wacdm"]["TT"]
ell = np.asarray(tt.get_ell_grid())
D_ell_jaxcapse = np.asarray(tt.get_Cl(jnp.asarray(params)))
assert np.array_equal(ell, np.arange(2, 9501))
assert D_ell_jaxcapse.shape == ell.shape and np.isfinite(D_ell_jaxcapse).all()

In [3]:
parameter_names = ("ln10As", "ns", "tau", "H0", "omega_b", "omega_c", "Mnu", "w0", "wa")
camb_params = dict(zip(parameter_names, params.tolist()))
D_ell_camb = camb_worker.compute_spectra(camb_params, int(ell[-1]))["TT_dense"]
assert D_ell_camb.shape == ell.shape and np.isfinite(D_ell_camb).all()

peak = np.max(np.abs(D_ell_camb))
residual = np.abs(D_ell_jaxcapse - D_ell_camb) / peak
print(f"median |jaxcapse - CAMB| / max|CAMB| = {np.median(residual):.3e}")
print(f"max    |jaxcapse - CAMB| / max|CAMB| = {np.max(residual):.3e}")
for sample_ell in (20, 200, 1000, 3000, 9500):
    i = sample_ell - 2
    print(f"ell={sample_ell:4d}: CAMB {D_ell_camb[i]:.8g}; jaxcapse {D_ell_jaxcapse[i]:.8g}")

median |jaxcapse - CAMB| / max|CAMB| = 6.302e-08
max    |jaxcapse - CAMB| / max|CAMB| = 1.613e-04
ell=  20: CAMB 906.80659; jaxcapse 906.68726
ell= 200: CAMB 5607.254; jaxcapse 5606.5362
ell=1000: CAMB 1061.8343; jaxcapse 1061.7744
ell=3000: CAMB 28.124634; jaxcapse 28.114989
ell=9500: CAMB 0.21499021; jaxcapse 0.2149375
